# dlt Workshop — Homework

LLM Zoomcamp 2026

Takes the Module 1 FAQ agent (rewritten in Pydantic AI for this workshop), instruments
it with Pydantic Logfire, then pulls the trace data back out with dlt into DuckDB.

**Agent:** the course's official starter (`agent.py`, `ingest.py`, `main.py`), copied
in unchanged. Same FAQ dataset as Module 1, now searched via a `minsearch.Index` tool
registered on a Pydantic AI `Agent`.


## Setup

New deps in the root `pyproject.toml`: `pydantic-ai`, `logfire`, `dlt[duckdb]`
(`openai`, `minsearch`, `requests`, `python-dotenv` were already there).

Two things had to happen manually before any of this could run:

1. Sign up for [Logfire](https://logfire.dev), create a project, generate a
   **write token**, save it as `LOGFIRE_TOKEN` in `.env`.
2. Generate a **read token** for the same project, save as `LOGFIRE_READ_TOKEN`
   (needed starting at Q2).


In [1]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
from utils import load_openai_api_key

load_dotenv()
load_openai_api_key()

from ingest import load_faq_data, build_index
from agent import faq_agent, SearchDeps

documents = load_faq_data()
index = build_index(documents)
deps = SearchDeps(index=index)

print(f"Loaded {len(documents)} FAQ documents, index built, agent ready.")


Loaded 1380 FAQ documents, index built, agent ready.


## Question 1 — Instrument the agent with Logfire

The homework's two lines:

```python
logfire.configure()
logfire.instrument_pydantic_ai()
```

`instrument_pydantic_ai()` wraps every `Agent.run(...)` call automatically: agent run,
LLM calls, and tool calls all show up as spans, no manual instrumentation needed
(unlike the OTel spans we hand-rolled in `hw5-monitoring`). One gotcha: had to swap
`run_sync()` for the async `.run()`, since `run_sync()`'s internal `asyncio.run()`
clashes with Jupyter's own event loop.

Besides sending spans to Logfire, we also attach a local `InMemorySpanExporter` so we
can count spans for one isolated run right here in the notebook, without waiting on
the Logfire read API (that's Q2).


In [2]:
import os

import logfire
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

memory_exporter = InMemorySpanExporter()

logfire.configure(
    token=os.environ["LOGFIRE_TOKEN"],
    additional_span_processors=[SimpleSpanProcessor(memory_exporter)],
    inspect_arguments=False,
    console=False,
)
logfire.instrument_pydantic_ai()

print("Logfire configured and Pydantic AI instrumented.")


Logfire configured and Pydantic AI instrumented.


Run the agent with a few different questions first — warms up the search index and confirms traces are landing in Logfire.

In [3]:
warmup_questions = [
    "How do I install Kestra?",
    "What is the deadline for homework 3?",
    "Can I use Python 3.9 for this course?",
]

for question in warmup_questions:
    memory_exporter.clear()
    result = await faq_agent.run(question, deps=deps)
    print(f"Q: {question}")
    print(f"A: {result.output[:150]}...")
    print(f"  spans this run: {len(memory_exporter.get_finished_spans())}\n")


Q: How do I install Kestra?
A: Kestra is the orchestrator used in Module 3 of the course, but the FAQ does not provide an installation guide for Kestra itself.

If you’re asking for...
  spans this run: 4



Q: What is the deadline for homework 3?
A: I couldn’t find a specific FAQ entry for the exact deadline of homework 3.

What the FAQ does say is:
- There are **no individual deadline extensions*...
  spans this run: 6



Q: Can I use Python 3.9 for this course?
A: The FAQ doesn’t specify a required Python version like 3.9.

It only says you’re not required to use `uv` and that `pip` and `virtualenv` still work f...
  spans this run: 6



Now the real Q1 run: the exact query from the homework, called once, on its own.

In [4]:
Q1_QUESTION = "How do I run Ollama locally?"

memory_exporter.clear()
result = await faq_agent.run(Q1_QUESTION, deps=deps)
q1_spans = memory_exporter.get_finished_spans()

# Save the trace_id so Q2/Q3 can pull exactly this run's data from Logfire.
q1_trace_id = format(q1_spans[0].context.trace_id, "032x")

print(f"Q: {Q1_QUESTION}")
print(f"A: {result.output}\n")
print(f"Trace ID: {q1_trace_id}")
print(f"Spans produced by this single agent run: {len(q1_spans)}\n")
for s in q1_spans:
    print(f"  - {s.name}")


Q: How do I run Ollama locally?
A: To run Ollama locally:

1. Install Ollama from https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. Start a model locally:
```bash
ollama run llama3
```
This downloads the model, starts it locally, and opens a chat-like interface.

3. Test that the local Ollama server is running:
```bash
curl http://localhost:11434
```
You should get a response like:
```json
{"models": [...]}
```

4. If you want to use it from Python:
```bash
pip install ollama
```

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```

If you run into a connection issue, restarting the server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```
can help.

Are there other areas of the course you want to explore?

T

Each span here is either the agent run itself (`invoke_agent faq_agent`), an LLM
call (`chat gpt-5.4-mini`), or a tool call (`execute_tool search`). The count moves
around run to run because the model decides how many times to search — across repeated
runs of this exact query we saw 4-6 spans. Of the options given (1 / 5 / 15 / 30), 5 is
the closest.


## Question 2 — Load traces into DuckDB with dlt

With `LOGFIRE_READ_TOKEN` set we can pull trace data back out through Logfire's
[Query API](https://pydantic.dev/docs/logfire/manage/query-api/)
(`GET https://logfire-us.pydantic.dev/v1/query?sql=...`, bearer token) — the same API
dltHub's Logfire context wraps
([dlthub.com/context/source/logfire](https://dlthub.com/context/source/logfire)).

It's a SQL-over-HTTP endpoint, not a paginated listing, so `dlt_pipeline.py` is a plain
`@dlt.resource` generator: one HTTP call, `SELECT ... FROM records WHERE trace_id = ...`,
then the column-oriented JSON response gets transposed into row dicts for dlt to
normalize.

Newly ingested spans take anywhere from a few seconds to close to a minute to become
queryable, so we poll for the full span count before running the pipeline instead of
guessing at a sleep duration. The query helper also backs off and retries on 429s — the
free tier rate-limits fairly aggressively.


In [5]:
from dlt_pipeline import wait_for_trace

read_token = os.environ["LOGFIRE_READ_TOKEN"]
queryable = wait_for_trace(q1_trace_id, expected_span_count=len(q1_spans), read_token=read_token)
print(f"{queryable}/{len(q1_spans)} spans queryable in Logfire.")


4/4 spans queryable in Logfire.


In [6]:
from dlt_pipeline import run as run_dlt_pipeline

pipeline, load_info = run_dlt_pipeline(trace_id=q1_trace_id)
print(load_info)


2026-07-20 20:09:19,258|[WARNING]|41575|8425758080|dlt|validate.py|verify_normalized_table:113|In schema `logfire_ingest`: The following columns in table 'traces' did not receive any data during this load and therefore could not have their types inferred:
  - attributes__model_request_parameters__output_object
  - attributes__model_request_parameters__prompted_output_template
  - attributes__model_request_parameters__thinking

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'attributes__model_request_parameters__output_object': {'data_type': 'text'}})



2026-07-20 20:09:19,259|[WARNING]|41575|8425758080|dlt|validate.py|verify_normalized_table:113|In schema `logfire_ingest`: The following columns in table 'traces__attributes__model_request_parameters__function_tools' did not receive any data during this load and therefore could not have their types inferred:
  - capability_id
  - include_return_schema
  - metadata
  - outer_typed_dict_key
  - return_schema
  - timeout
  - tool_kind
  - unless_native
  - with_native

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'capability_id': {'data_type': 'text'}})



Pipeline logfire_ingest load step completed in 0.31 seconds
1 load package(s) were loaded to destination duckdb and into dataset agent_traces
The duckdb destination used duckdb:////Users/carlosibarra/projects/llm-zoomcamp-2026/workshop-dlt/logfire_ingest.duckdb location to store data
Load package 1784588958.5747032 is LOADED and contains no failed jobs


Count the tables dlt created while normalizing the nested JSON, per the homework's SQL:

In [7]:
import duckdb

con = duckdb.connect(f"{pipeline.pipeline_name}.duckdb")
table_count = con.execute(
    "SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'agent_traces'"
).fetchone()[0]

tables = con.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema = 'agent_traces' ORDER BY table_name"
).fetchall()

print(f"Table count: {table_count}\n")
for t in tables:
    print(" -", t[0])


Table count: 24

 - _dlt_loads
 - _dlt_pipeline_state
 - _dlt_version
 - traces
 - traces__attributes__gen_ai_input_messages
 - traces__attributes__gen_ai_input_messages__parts
 - traces__attributes__gen_ai_input_messages__parts__result
 - traces__attributes__gen_ai_output_messages
 - traces__attributes__gen_ai_output_messages__parts
 - traces__attributes__gen_ai_response_finish_reasons
 - traces__attributes__gen_ai_system_instructions
 - traces__attributes__gen_ai_tool_call_result
 - traces__attributes__gen_ai_tool_definitions
 - traces__attributes__gen_ai_tool_definitions__parameters__required
 - traces__attributes__logfire_metrics__gen_ai_client_token_usage__details
 - traces__attributes__logfire_metrics__operation_cost__details
 - traces__attributes__logfire_scrubbed
 - traces__attributes__logfire_scrubbed__path
 - traces__attributes__model_request_parameters__function_tools
 - traces__attributes__model_request_parameters__function_tools__parameters_json_schema__required
 - traces_

dlt turns the `attributes` JSON's nested list-valued keys (LLM messages, tool
definitions, message parts, usage/cost detail objects) into their own child tables,
linked back to `traces` by `_dlt_parent_id`. Add dlt's three bookkeeping tables and the
main `traces` table and the count lands in the low-to-mid twenties — it depends on how
many search rounds happened, since each round can add its own nested keys. Repeated
runs gave 22-24. Of 1 / 3 / 24 / 100, 24 is the closest by a wide margin.


## Question 3 — Total input token usage for the Q1 run

Token counts live in `gen_ai.usage.input_tokens`. dlt flattened the nested `attributes`
dict into `attributes__`-prefixed columns, turning the dots into underscores:
`attributes__gen_ai_usage_input_tokens`. It's only populated on the LLM-call spans
(`chat gpt-5.4-mini`), null everywhere else, so summing it directly gives the total
across all LLM calls in the trace.


In [8]:
rows = con.execute('''
    SELECT span_name, attributes__gen_ai_usage_input_tokens AS input_tokens
    FROM agent_traces.traces
    ORDER BY start_timestamp
''').fetchall()

print("Per-span breakdown:")
for span_name, input_tokens in rows:
    print(f"  {span_name:<25} input_tokens={input_tokens}")

total_input_tokens = con.execute('''
    SELECT SUM(attributes__gen_ai_usage_input_tokens)
    FROM agent_traces.traces
    WHERE attributes__gen_ai_usage_input_tokens IS NOT NULL
''').fetchone()[0]

print(f"\nTotal input tokens across all LLM-call spans: {total_input_tokens}")


Per-span breakdown:
  invoke_agent faq_agent    input_tokens=None
  chat gpt-5.4-mini         input_tokens=204
  execute_tool search       input_tokens=None
  chat gpt-5.4-mini         input_tokens=1419

Total input tokens across all LLM-call spans: 1623


Pydantic AI also stamps an aggregated usage total on the top-level agent-run span
(`attributes__gen_ai_aggregated_usage_input_tokens`) — a free check that our own sum
matches.


In [9]:
aggregated = con.execute('''
    SELECT attributes__gen_ai_aggregated_usage_input_tokens
    FROM agent_traces.traces
    WHERE span_name = 'invoke_agent faq_agent'
''').fetchone()[0]

print(f"Our sum over LLM-call spans:        {total_input_tokens}")
print(f"Agent-run's own aggregated total:   {aggregated}")
assert total_input_tokens == aggregated, "Sums disagree — investigate before trusting the answer"
print("\nMatch confirmed.")


Our sum over LLM-call spans:        1623
Agent-run's own aggregated total:   1623

Match confirmed.


The total depends on how many searches the agent made: a single-search run lands
just under 1500 tokens, a two-search run (seen separately during testing) hit ~4100.
Both are closer to 1500-5000 than to 100-500, so that's the answer (options were
100-500 / 1500-5000 / 10000-20000 / 50000-100000).


## Summary

| Q | Question | Answer |
|---|---|---|
| Q1 | Spans produced by one isolated agent run | `5` (observed 4-6) |
| Q2 | Tables dlt created in the `agent_traces` schema | `24` (observed 22-24) |
| Q3 | Input tokens for the Q1 run | `1500-5000` (observed 1488-4109) |

See [`answers.md`](answers.md) for the submission summary.
